In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import ast
from torch.utils.data import Dataset, DataLoader
from pytorch_tcn import TCN

In [2]:
df = pd.read_csv('final_data.csv')
df['text'] = df['text'].apply(ast.literal_eval)
df['next_word'] = df['next_word'].apply(ast.literal_eval)
x = df['text'].values
y = df['next_word'].values

x = [torch.tensor(i, dtype=torch.long) for i in x]
y = [torch.tensor(i, dtype=torch.long) for i in y]

In [3]:
class NextWordDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [4]:
dataset = NextWordDataset(x, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [5]:
class TCNNextWord(nn.Module):

    def __init__(self, vocab_size):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            128,
            padding_idx=0
        )

        self.tcn = TCN(
            num_inputs=128,
            num_channels=[128, 128, 128],
            kernel_size=3,
            dropout=0.2
        )

        self.fc = nn.Linear(128, vocab_size)

    def forward(self, x):

        # x shape:
        # (batch_size, seq_len)

        x = self.embedding(x)

        # TCN expects:
        # (batch, channels, seq_len)

        x = x.permute(0, 2, 1)

        x = self.tcn(x)

        # last timestep

        x = x[:, :, -1]

        out = self.fc(x)

        return out

In [6]:
vocab_size = 3040

In [7]:
model = TCNNextWord(vocab_size)

criterion = nn.CrossEntropyLoss(ignore_index=0)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

In [8]:
for epoch in range(20):

    total_loss = 0

    for batch_x, batch_y in dataloader:

        optimizer.zero_grad()

        outputs = model(batch_x)
        batch_y = batch_y.squeeze().long()
        loss = criterion(outputs, batch_y)
        

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: {total_loss:.4f}")

Epoch 1: 4926.7693
Epoch 2: 3127.4967
Epoch 3: 2914.5748
Epoch 4: 2783.3237
Epoch 5: 2725.7047
Epoch 6: 2683.8660
Epoch 7: 2635.1064
Epoch 8: 2616.6459
Epoch 9: 2592.2208
Epoch 10: 2564.6709
Epoch 11: 2552.3419
Epoch 12: 2535.6465
Epoch 13: 2534.1521
Epoch 14: 2502.4036
Epoch 15: 2491.6160
Epoch 16: 2470.7138
Epoch 17: 2460.1560
Epoch 18: 2458.3653
Epoch 19: 2439.2244
Epoch 20: 2424.9154


In [9]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)
for epoch in range(20):

    total_loss = 0

    for batch_x, batch_y in dataloader:

        optimizer.zero_grad()

        outputs = model(batch_x)
        batch_y = batch_y.squeeze().long()
        loss = criterion(outputs, batch_y)
        

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: {total_loss:.4f}")

Epoch 1: 2190.4722
Epoch 2: 2076.6881
Epoch 3: 2027.0709
Epoch 4: 1996.6655
Epoch 5: 1971.0069
Epoch 6: 1964.4304
Epoch 7: 1953.1000
Epoch 8: 1949.4334
Epoch 9: 1946.7837
Epoch 10: 1936.5663
Epoch 11: 1935.5772
Epoch 12: 1934.7258
Epoch 13: 1932.1268
Epoch 14: 1927.5258
Epoch 15: 1925.2785
Epoch 16: 1923.9839
Epoch 17: 1917.3081
Epoch 18: 1920.7514
Epoch 19: 1916.4450
Epoch 20: 1914.5602


In [10]:
import json
with open('word_to_id.json', 'r') as f:
    id = json.load(f)

with open('id_to_word.json', 'r') as f:
    word_id = json.load(f)
def predict_next_word(text):

    tokens = text.split()

    seq = [id[word] for word in tokens if word in id]

    seq = torch.tensor(seq).unsqueeze(0)

    pad_len = 19 - seq.shape[1]

    seq = nn.functional.pad(
        seq,
        (pad_len, 0),
        value=0
    )

    with torch.no_grad():

        pred = model(seq)

        pred_idx = torch.argmax(pred, dim=1).item()

    return word_id[str(pred_idx)]

predict_next_word("i hope this email")

'finds'

In [11]:
text = "i appreciate your valuable"
for i in range(1,3):
    result = predict_next_word(text)
    text = text+" "+result

In [12]:
text

'i appreciate your valuable insights into'

In [13]:
torch.save(model.state_dict(), "TCN_next_word.pth")